In [77]:
import h5py 
import numpy as np
import os 
import sys 
from pathlib import Path
from tqdm import tqdm

# Make f0 test dataset 

## In this notebook:
* Read & combine saved f0 traces in .npy files with sources in .h5 files
* save new dataset for tests

In [4]:
## Get file paths  

data_path = '/om2/user/msaddler/projects/ibmHearingAid/assets/data/datasets/JSIN_v3.00/nStim_20000/2000ms/rms_0.1/noiseSNR_-10_10/stimSR_20000/reverb_none/noise_all/JSIN_all_v3/subsets/valid_RQTTZB4C3TJJVLJUWDV72TYMC7S4MNHH'

validation_h5s = list(Path(data_path).glob('*.h5'))

f0_trace_files = list(Path('../').glob("JSIN_all_*.npy"))



In [11]:
f0_trace_files, validation_h5s

([PosixPath('../JSIN_all__run_000_RQTTZB4C3TJJVLJUWDV72TYMC7S4MNHH_traces.npy'),
  PosixPath('../JSIN_all__run_001_RQTTZB4C3TJJVLJUWDV72TYMC7S4MNHH_traces.npy'),
  PosixPath('../JSIN_all__run_002_RQTTZB4C3TJJVLJUWDV72TYMC7S4MNHH_traces.npy'),
  PosixPath('../JSIN_all__run_003_RQTTZB4C3TJJVLJUWDV72TYMC7S4MNHH_traces.npy'),
  PosixPath('../JSIN_all__run_004_RQTTZB4C3TJJVLJUWDV72TYMC7S4MNHH_traces.npy'),
  PosixPath('../JSIN_all__run_005_RQTTZB4C3TJJVLJUWDV72TYMC7S4MNHH_traces.npy'),
  PosixPath('../JSIN_all__run_006_RQTTZB4C3TJJVLJUWDV72TYMC7S4MNHH_traces.npy'),
  PosixPath('../JSIN_all__run_007_RQTTZB4C3TJJVLJUWDV72TYMC7S4MNHH_traces.npy'),
  PosixPath('../JSIN_all__run_008_RQTTZB4C3TJJVLJUWDV72TYMC7S4MNHH_traces.npy'),
  PosixPath('../JSIN_all__run_009_RQTTZB4C3TJJVLJUWDV72TYMC7S4MNHH_traces.npy'),
  PosixPath('../JSIN_all__run_010_RQTTZB4C3TJJVLJUWDV72TYMC7S4MNHH_traces.npy'),
  PosixPath('../JSIN_all__run_011_RQTTZB4C3TJJVLJUWDV72TYMC7S4MNHH_traces.npy'),
  PosixPath('../JSIN_all__ru

In [52]:
## Sanity check that files are matched 

# pick one file 

file_name = validation_h5s[0].stem

# Get file - short lists so list comp is fine
wanted_h5 = [path.as_posix() for path in validation_h5s if file_name in path.as_posix()][0]
wanted_npy = [path.as_posix() for path in f0_trace_files if file_name in path.as_posix()][0]


h5 = h5py.File(wanted_h5, 'r') 
f0s = np.load(wanted_npy, allow_pickle=True)

In [16]:
file_name in validation_h5s[0].as_posix()

True

In [69]:
np.where((f0s[:,1] != h5['sources']['signal']['speaker_int'])
                       & (f0s[:,2] != h5['sources']['signal']['word_int']))



(array([], dtype=int64),)

In [64]:
np.nanmean(np.stack(f0s[:,0]), axis=1).shape

/tmp/ipykernel_60070/3849342436.py:1: RuntimeWarning: Mean of empty slice
  np.nanmean(np.stack(f0s[:,0]), axis=1).shape


(16812,)

In [41]:
# demo sanity check 
for index in tqdm(range(len(f0s)), total=len(f0s)):
    speaker_int = h5['sources']['signal']['speaker_int'][index]
    word_int = h5['sources']['signal']['word_int'][index]
    if not (f0s[index][1] == speaker_int and f0s[index][2] == word_int):
        print(f"bad ix at {index}")

100%|██████████| 16812/16812 [00:07<00:00, 2282.56it/s]


In [81]:
## Combine global ix, local ix, f0 mean, and f0 var id large pd df 

dict_of_data = {'global_ix':[],
              'ix_in_parent_h5':[],
              'f0_mean':[],
              'f0_var':[],
              'parent_h5_file':[]}
bad_ixs = {}

global_ix = 0 

for file in tqdm(validation_h5s):
    file_name = file.stem
    # get wanted files
    wanted_h5 = [path.as_posix() for path in validation_h5s if file_name in path.as_posix()][0]
    wanted_npy = [path.as_posix() for path in f0_trace_files if file_name in path.as_posix()][0]
    # load files 
    f0s = np.load(wanted_npy, allow_pickle=True)
    h5 = h5py.File(wanted_h5, 'r') 
    
    for local_ix, (f0_trace, speaker, word) in tqdm(enumerate(f0s), total=len(f0s), position=0,leave=True):
        speaker_int = h5['sources']['signal']['speaker_int'][local_ix]
        word_int = h5['sources']['signal']['word_int'][local_ix]
        # sanity check that indexing matches across npy and h5 files
        if speaker != speaker_int and word != word_int:
            print(f"bad ix at {local_ix} in {file_name}")
            bad_ixs.append([file_name, local_ix])
        # save relevant data 
        dict_of_data['global_ix'].append(global_ix)
        dict_of_data['ix_in_parent_h5'].append(local_ix)
        dict_of_data['parent_h5_file'].append(file.as_posix())
        dict_of_data['f0_mean'].append(np.nanmean(f0_trace))
        dict_of_data['f0_var'].append(np.nanvar(f0_trace))
        # update global ix 
        global_ix +=1 
    h5.close()





  0%|          | 0/16812 [00:00<?, ?it/s]/tmp/ipykernel_60070/1538141454.py:32: RuntimeWarning: Mean of empty slice
  dict_of_data['f0_mean'].append(np.nanmean(f0_trace))
/tmp/ipykernel_60070/1538141454.py:33: RuntimeWarning: Degrees of freedom <= 0 for slice.
  dict_of_data['f0_var'].append(np.nanvar(f0_trace))
100%|██████████| 16812/16812 [00:11<00:00, 1482.54it/s]

100%|██████████| 16812/16812 [00:12<00:00, 1366.20it/s]

100%|██████████| 16812/16812 [00:11<00:00, 1516.60it/s]

100%|██████████| 16812/16812 [00:18<00:00, 899.60it/s] 

100%|██████████| 16812/16812 [00:19<00:00, 871.67it/s] 

100%|██████████| 16812/16812 [00:18<00:00, 908.18it/s] 

100%|██████████| 16812/16812 [00:16<00:00, 992.19it/s] 

100%|██████████| 16812/16812 [00:13<00:00, 1225.57it/s]

100%|██████████| 16812/16812 [00:14<00:00, 1192.91it/s]

100%|██████████| 16812/16812 [00:16<00:00, 1044.74it/s]

100%|██████████| 16812/16812 [00:15<00:00, 1074.50it/s]

100%|██████████| 16812/16812 [00:15<00:00, 1079.50it/s]

1

In [109]:
len(np.unique(dict_of_data['parent_h5_file']))

22

In [85]:
len(dict_of_data['ix_in_parent_h5'])

369864

In [88]:
dict_of_data['global_ix'][-10:]

[369854,
 369855,
 369856,
 369857,
 369858,
 369859,
 369860,
 369861,
 369862,
 369863]

In [89]:
global_ix

369864

In [90]:
import pandas as pd

In [110]:
f0_df = pd.DataFrame.from_dict(dict_of_data)

In [111]:
f0_df.head()

,global_ix,ix_in_parent_h5,f0_mean,f0_var,parent_h5_file
0,0,0,NaN,NaN,/om2/user/msaddler/projects/ibmHearingAid/asse...
1,1,1,99.793562,62.570930,/om2/user/msaddler/projects/ibmHearingAid/asse...
2,2,2,198.486185,1247.393175,/om2/user/msaddler/projects/ibmHearingAid/asse...
3,3,3,259.937106,5532.252984,/om2/user/msaddler/projects/ibmHearingAid/asse...
4,4,4,111.343313,192.142368,/om2/user/msaddler/projects/ibmHearingAid/asse...


In [112]:
f0_df.to_pickle('../all_f0_ix_mapping_w_mean_var_par_file.pdpkl')

In [129]:
val_f0_df = f0_df[f0_df.parent_h5_file.str.contains('JSIN_all__run_000')]
test_f0_df = f0_df[~f0_df.parent_h5_file.str.contains('JSIN_all__run_000')]

In [130]:
test_f0_df['global_ix'] -=  16812

/tmp/ipykernel_60070/4285670713.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_f0_df['global_ix'] -=  16812


In [131]:
test_f0_df.reset_index(inplace=True, drop=True)

In [132]:
test_f0_df

,global_ix,ix_in_parent_h5,f0_mean,f0_var,parent_h5_file
0,0,0,102.353081,82.255873,/om2/user/msaddler/projects/ibmHearingAid/asse...
1,1,1,134.316029,434.371444,/om2/user/msaddler/projects/ibmHearingAid/asse...
2,2,2,213.953238,396.365795,/om2/user/msaddler/projects/ibmHearingAid/asse...
3,3,3,119.734713,68.679034,/om2/user/msaddler/projects/ibmHearingAid/asse...
4,4,4,111.205615,142.702347,/om2/user/msaddler/projects/ibmHearingAid/asse...
...,...,...,...,...,...
353047,353047,16807,170.385700,116.065754,/om2/user/msaddler/projects/ibmHearingAid/asse...
353048,353048,16808,267.381220,590.290813,/om2/user/msaddler/projects/ibmHearingAid/asse...
353049,353049,16809,231.871023,1053.636787,/om2/user/msaddler/projects/ibmHearingAid/asse...
353050,353050,16810,191.424660,474.940281,/om2/user/msaddler/projects/ibmHearingAid/asse...


In [133]:
val_f0_df.to_pickle('../validation_f0_ix_mapping_w_mean_var_and_parent_file.pdpkl')

In [134]:
test_f0_df.to_pickle('../test_f0_ix_mapping_w_mean_var_and_parent_file.pdpkl')

In [135]:
demo_load = pd.read_pickle('../validation_f0_ix_mapping_w_mean_var_and_parent_file.pdpkl')

In [150]:
test_f0_df[test_f0_df.global_ix == 3515].f0_mean.item()

279.68788088130015

In [159]:
file = '/om2/user/msaddler/projects/ibmHearingAid/assets/data/datasets/JSIN_v3.00/nStim_20000/2000ms/rms_0.1/noiseSNR_-10_10/stimSR_20000/reverb_none/noise_all/JSIN_all_v3/subsets/valid_RQTTZB4C3TJJVLJUWDV72TYMC7S4MNHH/JSIN_all__run_001_RQTTZB4C3TJJVLJUWDV72TYMC7S4MNHH.h5'

test_f0_df[(test_f0_df.parent_h5_file == file)
           & (test_f0_df.ix_in_parent_h5==35)]



,global_ix,ix_in_parent_h5,f0_mean,f0_var,parent_h5_file
35,35,35,110.739112,125.635135,/om2/user/msaddler/projects/ibmHearingAid/asse...
